In [ ]:
language = "pt"

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(seconds=10):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (seconds * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=False))


In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

In [ ]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)


In [ ]:
import os

# Link direto: https://platform.openai.com/account/api-keys
os.environ['OPENAI_API_KEY'] = 'OPENAI_API_KEY_value'

In [ ]:
if language == "pt":
  fixed_prompt = """
  e me responda seguindo o seguinte  padrão de resposta:

  explicação breve: [texto curto e direto]
  3 opções de como fazer:
  1. [primeira opção]
  2. [segunda opção]
  3. [terceira opção]
  fontes:
  [lista de fontes]
  """
else:
  fixed_prompt = """
  Please respond following this response pattern:

  brief explanation: [short and direct text]
  3 options on how to do it:
  1. [first option]
  2. [second option]
  3. [third option]
  sources:
  [list of sources]
  """


In [ ]:
import openai
import os

openai.api_key = os.environ.get('OPENAI_API_KEY')

prompt_final = transcription + fixed_prompt

response = openai.ChatCompletion.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt_final}]
)

chatgpt_response = response.choices[0].message.content
print(chatgpt_response)


In [ ]:
!pip install gTTS

In [ ]:
from gtts import  gTTS

gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)

response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))